In [1]:
print('all ok')

all ok


## Use mem0 as a memory store

In [7]:
from autogen_ext.memory.mem0 import Mem0Memory
import os
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_core.memory import MemoryContent,MemoryMimeType
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

mem0_api_key=os.getenv("MEM0_API_KEY")

In [9]:
model_client=OpenAIChatCompletionClient(model='gpt-4o')

In [18]:
mem0_memory=Mem0Memory(
    user_id='tapas1234',
    is_cloud=True,
    api_key=mem0_api_key
)

In [21]:
async def main():
    try:

        print("Attempting to add memory to cloud.")
        messages=[
            {
                "role":"user", "content":"I am vegetarian food and I love cooking."
            },
            {
                "role":"assistant","content":"I will remember preferces is veg items."
            }
        ]

        mem0_memory._client.add(messages,user_id='tapas1234',metadata={"category":"food","type":"vegetarian"})
        messages=[
            {
                "role":"user", "content":"I want weather in metric unit."
            },
            {
                "role":"assistant","content":"I will remember preferces is metric unit for weather."
            }
        ]

        mem0_memory._client.add(messages,user_id='tapas1234',metadata={"category":"preferences","type":"unit"})

        messages=[
            {
                "role":"user", "content":"My faviourite color is blue"
            },
            {
                "role":"assistant","content":"I will remember color preferences is blue."
            }
        ]

        mem0_memory._client.add(messages,user_id='tapas1234',metadata={"category":"preferences","type":"color"})
        print("Memory added in the cloud")
    except Exception as e:
        print(e)


    agent=AssistantAgent(
        name="assistant",
        model_client=model_client,
        memory=[mem0_memory],
        system_message="You are a helpful assistant that remember user preferences and use them to provide responses."
        )

    try:
        result=await agent.run(task="What type of food I like to eat?")
        print(result.messages[-1].content)
    except Exception as e:
        print(f"Erron in vcalling agent. {e}")
        return e


In [22]:
await main()

Attempting to add memory to cloud.
Memory added in the cloud
You mentioned that you are vegetarian, so it's likely you enjoy plant-based dishes. If you have any specific preferences or favorite vegetarian dishes, feel free to share them!


In [23]:
async def get_weather(city: str, units: str = "imperial") -> str:
    if units == "imperial":
        return f"The weather in {city} is 73 °F and Sunny."
    elif units == "metric":
        return f"The weather in {city} is 23 °C and Sunny."
    else:
        return f"Sorry, I don't know the weather in {city}."

In [24]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_core.memory import MemoryContent, MemoryMimeType
from autogen_ext.memory.mem0 import Mem0Memory
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Initialize Mem0 cloud memory (requires API key)
# For local deployment, use is_cloud=False with appropriate config
mem0_memory = Mem0Memory(
    is_cloud=True,
    limit=5,  # Maximum number of memories to retrieve
    api_key=mem0_api_key,
    user_id="tapas123"
)

# Add user preferences to memory
await mem0_memory.add(
    MemoryContent(
        content="The weather should be in metric units",
        mime_type=MemoryMimeType.TEXT,
        metadata={"category": "preferences", "type": "units"},
    )
)

await mem0_memory.add(
    MemoryContent(
        content="Meal recipe must be vegan",
        mime_type=MemoryMimeType.TEXT,
        metadata={"category": "preferences", "type": "dietary"},
    )
)

# Create assistant with mem0 memory
assistant_agent = AssistantAgent(
    name="assistant_agent",
    model_client=OpenAIChatCompletionClient(
        model="gpt-4o-2024-08-06",
    ),
    tools=[get_weather],
    memory=[mem0_memory],
)

# Ask about the weather
stream = assistant_agent.run_stream(task="What are my dietary preferences?")
await Console(stream)


---------- TextMessage (user) ----------
What are my dietary preferences?
---------- MemoryQueryEvent (assistant_agent) ----------
[MemoryContent(content='Meal recipe must be vegan', mime_type='text/plain', metadata={'type': 'dietary', 'category': 'preferences', 'score': 0.4950689224919358, 'created_at': datetime.datetime(2025, 8, 18, 6, 6, 28, 915500, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))), 'updated_at': datetime.datetime(2025, 8, 18, 6, 6, 28, 964443, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200)))}), MemoryContent(content='User prefers weather information in metric units', mime_type='text/plain', metadata={'type': 'units', 'category': 'preferences', 'score': 0.4101268175282128, 'created_at': datetime.datetime(2025, 8, 18, 6, 6, 22, 887472, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))), 'updated_at': datetime.datetime(2025, 8, 18, 6, 6, 22, 985366, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds

TaskResult(messages=[TextMessage(id='4e372af5-2436-48f1-8b51-620e1261973e', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 18, 13, 6, 30, 667691, tzinfo=datetime.timezone.utc), content='What are my dietary preferences?', type='TextMessage'), MemoryQueryEvent(id='7849f30e-8c73-47e7-9d9a-3506be20658e', source='assistant_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 18, 13, 6, 31, 120788, tzinfo=datetime.timezone.utc), content=[MemoryContent(content='Meal recipe must be vegan', mime_type='text/plain', metadata={'type': 'dietary', 'category': 'preferences', 'score': 0.4950689224919358, 'created_at': datetime.datetime(2025, 8, 18, 6, 6, 28, 915500, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))), 'updated_at': datetime.datetime(2025, 8, 18, 6, 6, 28, 964443, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200)))}), MemoryContent(content='User prefers weather information in metric units